# 2. Convexidade — Jacobiana e Hessiana analíticas

Antes de otimizar, classifica-se cada peça da PNL. As derivadas saem do SymPy,
sem diferenças finitas. A figura do C-space é a “cara” da região viável em
\((q_1,q_2)\); a PNL em \(\mathbb{R}^{101}\) é um levantamento dessa geometria
com \(t_f\) e as acelerações.


In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
for cand in [here, here.parent, here / "T2", here.parent / "T2"]:
    if (cand / "src" / "parametros.py").exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break
from src.ambiente import preparar
T2 = preparar()
print("T2 =", T2)


In [ ]:
import numpy as np
from IPython.display import display, Math
from src.convexidade import (
    classificar_autovalores, grade_cspace, hessiana_folga, hessiana_objetivo,
    hessiana_residual, jacobiana_folga, latex_pecas, numerica_folga,
)
from src.parametros import P
from src.paths import figuras_dir
from src.simulacao import figura_cspace

lx = latex_pecas()
display(Math(r"\text{FK }x_P=" + lx["fk_x"]))
display(Math(r"\text{FK }y_P=" + lx["fk_y"]))
display(Math(r"g(\mathbf{q},\zeta)=" + lx["g"]))
display(Math(r"\nabla_q g=" + lx["J_g"]))
display(Math(r"\nabla^2_q g=" + lx["H_g"]))


## Classificação das peças

| Peça | Forma | Convexidade |
|------|--------|-------------|
| \(J=t_f\) | linear | convexa (Hessiana nula) |
| caixas de \(t_f\) e \(a\) | intervalos | convexas |
| residual \(v_{k+1}-v_k-(t_f/K)a_k\) | bilinear | nem convexo nem côncavo |
| \(g(\mathbf{q},\zeta)=R^2-\|\mathbf{r}-\mathbf{r}_0\|^2\) | trigonométrica | Hessiana indefinida |
| \(\{g\le 0\}\) no plano das juntas | complementar de um disco via FK | **não convexo** |


In [ ]:
print("Hessiana de J = tf:")
print(hessiana_objetivo())

Hr, vars_r = hessiana_residual()
print("\nHessiana do residual bilinear nas variáveis", vars_r)
print(Hr)

g, J, H = numerica_folga(0.7, -0.4, zeta_val=1.0)
w, tipo = classificar_autovalores(H)
print("\nEm (q1,q2)=(0.7,-0.4), zeta=1:")
print("g =", g)
print("J =", J)
print("H =\n", H)
print("autovalores", w, "→", tipo)


In [ ]:
from src.transcricao import integrar
q1g, q2g, G = grade_cspace(n1=201, n2=201)
caminho = figuras_dir() / "cspace_regiao.png"
figura_cspace(None, caminho, q1g=q1g, q2g=q2g, G=G)
print("figura:", caminho)
print("fração do retângulo em colisão:", float((G > 0).mean()))
print("Conclusão: a PNL não é convexa. Mínimos locais distintos são esperados;")
print("as cinco partidas do notebook 03 não são cosmética.")
